# From policy gradients to RLHF, DPO and GRPO, on toy problems

The lecture builds one idea step by step: **sample, score, then raise the log-probability of what you did, weighted by how good it was.** This notebook runs every step on problems small enough to see completely:

| Part | What you build | Slides it follows |
|---|---|---|
| 1 | The discounted return $G_t$ on the grid world | MDP setup, policies, Q-learning example |
| 2 | A toy "LLM" policy that answers `a+b=` with one token, and a verifier | the RL-to-LLM dictionary, RL with verifiable rewards |
| 3 | The REINFORCE loss, and why a baseline helps | policy gradient in one line, REINFORCE, the baseline check |
| 4 | A Bradley–Terry reward model fit on preference pairs | reward model, its loss |
| 5 | The KL-penalised optimum, and reward hacking | KL penalty, RLHF objective, DPO closed form |
| 6 | The DPO loss | DPO |
| 7 | GRPO: group-relative advantages and the PPO clip | "what multiplies $\nabla\log\pi$", PPO, GRPO in our notation |

**How it works.** Each part has a short explanation, then a `TODO` cell with a small function to write, then a check cell. Until you fill a `TODO`, its check cell prints a reminder and skips, so the notebook always runs top to bottom. Everything runs on a CPU in a few minutes and uses only synthetic data.

**Notation (as on the slides).** State $s_t$, action $a_t$, reward $r_t$, discount $\gamma$, return $G_t$, policy $\pi_\theta(a\mid s)$, step size $\alpha$. For language models: prompt $x$, answer $y$, reward $r(x, y)$, reference policy $\pi_{\text{ref}}$, KL weight $\beta$, group size $G$, answer index $g$, ratio $\rho_t$, clip range $\varepsilon$.

In [1]:
import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.set_num_threads(2)
torch.manual_seed(0)
np.random.seed(0)
T_START = time.time()

def attempt(fn, *args, **kwargs):
    """Run a TODO function; if it is not written yet, say so and return None instead of crashing."""
    try:
        return fn(*args, **kwargs)
    except NotImplementedError:
        print(f"[TODO] {fn.__name__} is not implemented yet: skipping this check.")
        return None

print("torch", torch.__version__)

torch 2.9.1


## Part 1. The return $G_t$

The return from step $t$ adds up all later rewards, each discounted by $\gamma$ once per step of delay:

$$G_t = r_t + \gamma\, r_{t+1} + \gamma^2 r_{t+2} + \dots = \sum_{k \ge 0} \gamma^k\, r_{t+k}.$$

A useful fact: $G_t = r_t + \gamma\, G_{t+1}$. So you can compute all returns in one pass, **from the end backwards**.

**The grid world from the slides.** A 3-by-3 grid, cells written $\langle x, y\rangle$ with $\langle 0,0\rangle$ at the top left. The chest at $\langle 2,0\rangle$ gives reward 100 and ends the episode; every other move gives 0; $\gamma = 0.5$. A shortest path from the bottom-left cell $\langle 0,2\rangle$ takes four moves, so its rewards are $[0, 0, 0, 100]$.

In [2]:
# TODO 1: discounted returns.
def discounted_returns(rewards, gamma):
    """rewards: list of floats r_0, ..., r_{T-1}. Return the list G_0, ..., G_{T-1}.
    Hint: loop over the rewards backwards, using G_t = r_t + gamma * G_{t+1}."""
    raise NotImplementedError

In [3]:
G = attempt(discounted_returns, [0, 0, 0, 100], 0.5)
if G is not None:
    print("shortest path from <0,2>:  G_t =", G)
    assert np.allclose(G, [12.5, 25, 50, 100])
    detour = discounted_returns([0, 0, 0, 0, 100], 0.5)
    print("a one-move detour:          G_0 =", detour[0])
    assert np.isclose(detour[0], 6.25)
    # compare with the definition, on random rewards
    rr = np.random.default_rng(0).normal(size=7)
    brute = [sum(0.9 ** k * rr[t + k] for k in range(len(rr) - t)) for t in range(len(rr))]
    assert np.allclose(discounted_returns(list(rr), 0.9), brute)
    print("matches the sum definition: ok")

[TODO] discounted_returns is not implemented yet: skipping this check.


These are the numbers in the converged Q-table on the slides: 100, 50, 25, 12.5 along a shortest path, and 6.25 when the first move is a detour. Each extra step before the reward costs a factor $\gamma$.

For a language model the episode is one answer and the reward arrives once, at the end (with $\gamma = 1$ in practice). Then $G_t$ is the same number for every token of the answer. That is why GRPO, in Part 7, gives all tokens of an answer the same advantage.

## Part 2. A toy LLM as a policy, and a verifier

The slides' dictionary: state = prompt $x$ (plus tokens so far), action = next token, policy = the softmax at the last position, reward $r(x, y)$ = once, at the end. Here we keep only what matters for RL:

* **Prompts**: the 100 strings `a+b=` with $a, b \in \{0, \dots, 9\}$.
* **Answer**: a single token from a vocabulary of 19 tokens, `"0"` to `"18"`. So one episode is one action, a *contextual bandit*.
* **Policy** $\pi_\theta(y \mid x)$: a small MLP from the one-hot digits $a$, $b$ to 19 logits.
* **Verifier**: $r(x, y) = 1$ if $y = a + b$, else 0. This is a *verifiable reward*: no reward model needed.

**A starting point like SFT.** RL usually starts from a supervised fine-tuned model. We imitate one: train the policy with cross-entropy on noisy demonstrations, where the label is correct 25% of the time, **one too high** 50% of the time, and a random token the other 25%. This policy becomes our reference $\pi_{\text{ref}}$. Its most likely answer is often off by one.

We measure a policy two ways: the **expected reward** $\mathbb{E}_x\, \pi_\theta(a{+}b \mid x)$ (the chance that one sampled answer is correct, averaged over the 100 prompts), and the **greedy accuracy** (is the most likely token correct?).

In [4]:
VOCAB = 19
A, B = torch.meshgrid(torch.arange(10), torch.arange(10), indexing="ij")
PROMPTS = torch.stack([A.flatten(), B.flatten()], 1)   # (100, 2): the prompt "a+b="
ANSWER = PROMPTS.sum(1)                                  # the correct token for each prompt

def verify(prompts, y):
    """Verifiable reward: 1.0 if the answer token equals a + b, else 0.0."""
    return (y == prompts.sum(1)).float()

class Policy(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20, hidden), nn.Tanh(), nn.Linear(hidden, VOCAB))
    def forward(self, prompts):                          # logits over the 19 answer tokens
        x = torch.cat([F.one_hot(prompts[:, 0], 10), F.one_hot(prompts[:, 1], 10)], 1).float()
        return self.net(x)

@torch.no_grad()
def evaluate(policy, ref=None):
    probs = F.softmax(policy(PROMPTS), -1)
    out = {"expected reward": probs[torch.arange(100), ANSWER].mean().item(),
           "greedy acc": (probs.argmax(-1) == ANSWER).float().mean().item()}
    if ref is not None:
        out["KL to ref"] = kl_divergence(probs, F.softmax(ref(PROMPTS), -1)).mean().item()
    return out

def kl_divergence(p, q):
    """KL(p || q) = sum_a p(a) log(p(a)/q(a)) along the last axis. 0 iff p == q; not symmetric."""
    return (p * (torch.log(p + 1e-12) - torch.log(q + 1e-12))).sum(-1)

# --- the SFT-like reference policy, from noisy demonstrations
g = torch.Generator().manual_seed(0)
def noisy_demo(prompts):
    u = torch.rand(len(prompts), generator=g)
    y = prompts.sum(1).clone()
    y[(u >= 0.25) & (u < 0.75)] += 1                                   # one too high
    rnd = u >= 0.75
    y[rnd] = torch.randint(VOCAB, (int(rnd.sum()),), generator=g)       # random token
    return y.clamp(0, VOCAB - 1)

torch.manual_seed(0)
ref_policy = Policy()
opt = torch.optim.Adam(ref_policy.parameters(), lr=1e-2)
for step in range(400):
    idx = torch.randint(100, (256,), generator=g)
    loss = F.cross_entropy(ref_policy(PROMPTS[idx]), noisy_demo(PROMPTS[idx]))
    opt.zero_grad(); loss.backward(); opt.step()
for p_ in ref_policy.parameters():
    p_.requires_grad_(False)

REF_EVAL = evaluate(ref_policy)
print("reference policy:", {k: round(v, 3) for k, v in REF_EVAL.items()})
p0 = F.softmax(ref_policy(PROMPTS[[37]]), -1)[0]
print("prompt 3+7=  top answers:", [(int(i), round(float(p0[i]), 2)) for i in p0.argsort(descending=True)[:3]])
off_by_one = (F.softmax(ref_policy(PROMPTS), -1).argmax(-1) == (ANSWER + 1).clamp(max=VOCAB - 1)).float().mean().item()
print(f"share of prompts whose most likely answer is one too high: {off_by_one:.2f}")
assert REF_EVAL["expected reward"] < 0.4 and REF_EVAL["greedy acc"] < 0.5 and off_by_one > 0.5

reference policy: {'expected reward': 0.276, 'greedy acc': 0.21}
prompt 3+7=  top answers: [(11, 0.44), (10, 0.22), (12, 0.07)]
share of prompts whose most likely answer is one too high: 0.80


## Part 3. REINFORCE and the baseline

The policy gradient (Williams, 1992), as on the slides:

$$\nabla_\theta J \approx \frac{1}{N}\sum_{n=1}^{N} W_n\, \nabla_\theta \log \pi_\theta(a_n \mid s_n).$$

Every method in the lecture uses this gradient; they differ only in the **weight** $W$: the return $G_t$ (REINFORCE), $G_t - b(s_t)$ (with a baseline), an advantage from a critic (actor-critic, PPO), or a group-normalised reward (GRPO).

In code we never write the gradient by hand. We write a **surrogate loss** whose gradient is minus that estimate, and let autograd do the rest:

$$\mathcal{L}(\theta) = -\frac{1}{N}\sum_n W_n \log \pi_\theta(a_n \mid s_n), \qquad W_n \text{ treated as a constant.}$$

The slides also say: with data tokens instead of samples and $W = 1$, this is exactly the next-token cross-entropy. The check cell tests that.

In [5]:
# TODO 2: the REINFORCE surrogate loss.
def reinforce_surrogate(logits, actions, weights):
    """logits: (N, V) policy logits; actions: (N,) long, the actions taken; weights: (N,) the W_n.
    Return the scalar -(1/N) * sum_n W_n * log pi(a_n | s_n). No gradient may flow through the weights.
    Hint: F.log_softmax, then .gather(1, actions[:, None])."""
    raise NotImplementedError

In [6]:
torch.manual_seed(1)
lg = torch.randn(6, VOCAB, requires_grad=True)
acts = torch.randint(VOCAB, (6,))
w = torch.randn(6, requires_grad=True)
L = attempt(reinforce_surrogate, lg, acts, w)
if L is not None:
    L.backward()
    # the gradient w.r.t. the logits of log-softmax is (one-hot - softmax)
    expected = -(F.one_hot(acts, VOCAB) - F.softmax(lg, -1)).detach() * w.detach()[:, None] / 6
    assert torch.allclose(lg.grad, expected, atol=1e-6), "gradient is not -(onehot - softmax) * W / N"
    assert w.grad is None or torch.all(w.grad == 0), "gradient leaked into the weights: detach them"
    ce = F.cross_entropy(lg, acts)
    assert torch.isclose(reinforce_surrogate(lg, acts, torch.ones(6)), ce)
    print("gradient = -(onehot - softmax) * W / N: ok")
    print(f"with W = 1 the surrogate equals cross-entropy: {ce.item():.4f}: ok")

[TODO] reinforce_surrogate is not implemented yet: skipping this check.


### Does a baseline bias the gradient? (the check slide)

For a fixed state $s$ and a baseline $b(s)$ that does not depend on the action,
$\sum_a \pi_\theta(a\mid s)\, b(s)\, \nabla_\theta \log \pi_\theta(a\mid s) = b(s)\, \nabla_\theta \sum_a \pi_\theta(a \mid s) = b(s)\,\nabla_\theta 1 = 0.$

The next cell checks this exactly on the reference policy (a sum over all 19 actions, not a sample). Then it measures what the baseline buys: the **variance** of the sampled gradient. We draw many batches of 128 (prompt, answer) samples from $\pi_{\text{ref}}$ and compare the spread of the gradient estimate for $W = r$ and for $W = r - V(s)$, where $V(s) = \pi_{\text{ref}}(a{+}b \mid s)$ is the exact value of the prompt.

In [7]:
if attempt(reinforce_surrogate, torch.zeros(1, VOCAB), torch.zeros(1, dtype=torch.long), torch.ones(1)) is not None:
    pol = Policy(); pol.load_state_dict(ref_policy.state_dict())
    params = list(pol.parameters())
    flat = lambda gs: torch.cat([g_.flatten() for g_ in gs])

    # (1) exact expectation of b * grad log pi over all actions, for one prompt
    s = PROMPTS[[37]]
    logp_all = F.log_softmax(pol(s), -1)[0]
    b = 0.7
    exact = torch.autograd.grad((logp_all.exp().detach() * b * logp_all).sum(), params)
    print(f"|| E_a[ b * grad log pi ] ||  = {flat(exact).norm():.2e}   (zero up to rounding)")
    assert flat(exact).norm() < 1e-5

    # (2) variance of the sampled gradient, with and without the baseline
    def grad_estimate(use_baseline, gen):
        idx = torch.randint(100, (128,), generator=gen)
        s = PROMPTS[idx]
        logits = pol(s)
        y = torch.multinomial(F.softmax(logits, -1).detach(), 1, generator=gen).squeeze(1)
        r = verify(s, y)
        V = F.softmax(logits, -1).detach()[torch.arange(128), s.sum(1)]
        W = r - V if use_baseline else r
        return flat(torch.autograd.grad(reinforce_surrogate(logits, y, W), params))

    var = {}
    for use_b in (False, True):
        gen = torch.Generator().manual_seed(123)
        gs = torch.stack([grad_estimate(use_b, gen) for _ in range(300)])
        var[use_b] = gs.var(0).sum().item()
        print(f"W = {'r - V(s)' if use_b else 'r       '}: total gradient variance {var[use_b]:.4f}, "
              f"mean-gradient norm {gs.mean(0).norm():.4f}")
    print(f"variance ratio (no baseline / baseline) = {var[False] / var[True]:.1f}")
    assert var[True] < var[False]

[TODO] reinforce_surrogate is not implemented yet: skipping this check.


Both weightings aim at the same expected gradient, but the baseline gives a less noisy estimate (the printout shows the ratio). The baseline asks "how much **better than expected** was this answer?" instead of "was it good?". With $b(s) = V(s)$ the weight is the advantage $A(s, a) = Q(s,a) - V(s)$.

The exact $V(s)$ is a luxury of this toy. Real systems must estimate it: actor-critic and PPO train a second network, the critic; GRPO, in Part 7, uses the mean reward of a group of answers to the same prompt.

## Part 4. A reward model from preference pairs (Bradley–Terry)

For "be helpful" there is no verifier. RLHF instead asks people which of two answers is better, and fits a **reward model** $r_\phi(x, y)$ to those choices. The Bradley–Terry model (Bradley & Terry, 1952) says

$$P(y_w \succ y_l \mid x) = \sigma\big(r_\phi(x, y_w) - r_\phi(x, y_l)\big), \qquad \mathcal{L}(\phi) = -\,\mathbb{E}\Big[\log \sigma\big(r_\phi(x, y_w) - r_\phi(x, y_l)\big)\Big].$$

It is logistic regression on pairs, and it only sees **differences** of rewards.

**A synthetic preference dataset.** One prompt, $K = 40$ candidate answers. Answer $k$ has a *quality* $q_k \in [0, 1]$ and a *length* $\ell_k$ between 20 and 400 words. The hidden "true human" reward likes quality, and likes length **only up to about 200 words**:

$$r^*(k) = 3\, q_k + 2\,\frac{\ell_k}{200} - \Big(\frac{\ell_k}{200}\Big)^2.$$

The reference policy prefers short answers, $\pi_{\text{ref}}(k) \propto e^{-\ell_k/80}$. We draw 2000 pairs of answers from $\pi_{\text{ref}}$ and label each pair by a Bradley–Terry coin flip with $r^*$. The reward model is linear, $r_\phi(k) = w_q\, q_k + w_\ell\, \ell_k / 400$.

In [8]:
K = 40
rng = np.random.default_rng(0)
quality = torch.tensor(rng.uniform(0, 1, K), dtype=torch.float32)
length = torch.linspace(20, 400, K)
true_reward = 3 * quality + 2 * (length / 200) - (length / 200) ** 2
log_pi_ref = torch.log_softmax(-length / 80, 0)
features = torch.stack([quality, length / 400], 1)          # (K, 2)

gp = torch.Generator().manual_seed(1)
def sample_pairs(n):
    i = torch.multinomial(log_pi_ref.exp(), n, replacement=True, generator=gp)
    j = torch.multinomial(log_pi_ref.exp(), n, replacement=True, generator=gp)
    keep = i != j
    i, j = i[keep], j[keep]
    i_wins = torch.rand(len(i), generator=gp) < torch.sigmoid(true_reward[i] - true_reward[j])
    return torch.where(i_wins, i, j), torch.where(i_wins, j, i)     # (winner y_w, loser y_l)

YW, YL = sample_pairs(2000)
YW_test, YL_test = sample_pairs(2000)
covered = (length[torch.cat([YW, YL])] <= 200).float().mean().item()
print(f"{len(YW)} training pairs; share of answers in them with length <= 200 words: {covered:.2f}")

1877 training pairs; share of answers in them with length <= 200 words: 0.90


In [9]:
# TODO 3: the Bradley-Terry reward-model loss.
def bt_loss(r_w, r_l):
    """r_w, r_l: (N,) rewards of the preferred and the rejected answers.
    Return the mean of -log sigmoid(r_w - r_l). Hint: F.logsigmoid is the numerically stable log(sigmoid)."""
    raise NotImplementedError

In [10]:
v = attempt(bt_loss, torch.tensor([2.0, 0.0]), torch.tensor([0.0, 0.0]))
if v is not None:
    assert math.isclose(v.item(), (-math.log(torch.sigmoid(torch.tensor(2.0)).item()) + math.log(2)) / 2, rel_tol=1e-5)
    rw, rl = torch.randn(50), torch.randn(50)
    assert torch.isclose(bt_loss(rw, rl), bt_loss(rw + 5, rl + 5)), "BT should only see differences"
    print(f"equal rewards give log 2 = {bt_loss(torch.zeros(3), torch.zeros(3)).item():.4f}; "
          "adding a constant changes nothing: ok")

    # fit the linear reward model
    w_rm = torch.zeros(2, requires_grad=True)
    opt = torch.optim.Adam([w_rm], lr=0.05)
    for step in range(600):
        r = features @ w_rm
        loss = bt_loss(r[YW], r[YL])
        opt.zero_grad(); loss.backward(); opt.step()
    w_rm = w_rm.detach()
    rm_reward = features @ w_rm
    acc = (rm_reward[YW_test] > rm_reward[YL_test]).float().mean().item()
    print(f"reward model: w_q = {w_rm[0]:.2f}, w_len = {w_rm[1]:.2f}; "
          f"agrees with held-out human choices on {acc:.1%} of pairs")
    assert w_rm[0] > 0 and w_rm[1] > 0 and acc > 0.6

    fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
    ax[0].bar(length, log_pi_ref.exp(), width=8, color="gray")
    ax[0].set_title(r"$\pi_{ref}$ over answers (by length)"); ax[0].set_xlabel("length (words)")
    order = length.argsort()
    ax[1].plot(length, true_reward - 3 * quality, label="true reward, length part")
    ax[1].plot(length, rm_reward - w_rm[0] * quality, label="reward model, length part")
    ax[1].axvline(200, ls=":", c="k"); ax[1].set_xlabel("length (words)"); ax[1].legend()
    ax[1].set_title("what each reward says about length")
    plt.tight_layout(); plt.show()

[TODO] bt_loss is not implemented yet: skipping this check.


The reward model gets quality right, and it also learned "**longer is better**". On the data it saw, that is mostly true: most answers drawn from $\pi_{\text{ref}}$ are 200 words or shorter (the printout gives the share), and there adding length does help. Beyond 200 words the true reward falls, but the linear reward model keeps climbing. No preference data lives there to correct it. A policy that optimises this reward model hard will go exactly there.

## Part 5. The KL penalty, and reward hacking

The RLHF objective on the slides:

$$\max_\pi\; \mathbb{E}_{y \sim \pi}\big[r_\phi(x, y)\big] \;-\; \beta\, D_{\mathrm{KL}}\big(\pi \,\|\, \pi_{\text{ref}}\big).$$

The DPO slides use its closed-form solution (derived in Rafailov et al., 2023, and known earlier in KL-regularised control):

$$\pi^*_\beta(y \mid x) = \frac{1}{Z(x)}\, \pi_{\text{ref}}(y \mid x)\, \exp\!\big(r_\phi(x, y)/\beta\big).$$

With only $K = 40$ answers we can compute it exactly. Large $\beta$: stay at $\pi_{\text{ref}}$. Small $\beta$: put all mass on the reward model's favourite answer.

In [11]:
# TODO 4: the KL-regularised optimal policy.
def kl_regularized_optimum(log_pi_ref, reward, beta):
    """log_pi_ref: (K,) log-probabilities of the reference policy; reward: (K,); beta > 0.
    Return the (K,) probabilities pi*(y) proportional to pi_ref(y) * exp(reward(y) / beta).
    Hint: work in log space and use torch.softmax for the normalisation Z."""
    raise NotImplementedError

In [12]:
def rlhf_objective(pi, reward, beta):
    return (pi * reward).sum() - beta * kl_divergence(pi, log_pi_ref.exp())

pi_b = attempt(kl_regularized_optimum, log_pi_ref, true_reward, 0.5)
if pi_b is not None:
    assert torch.isclose(pi_b.sum(), torch.tensor(1.0))
    assert torch.allclose(kl_regularized_optimum(log_pi_ref, true_reward, 1e4), log_pi_ref.exp(), atol=1e-3)
    assert kl_regularized_optimum(log_pi_ref, true_reward, 1e-3).argmax() == true_reward.argmax()
    # no random perturbation of pi* scores higher on the objective
    best = rlhf_objective(pi_b, true_reward, 0.5)
    gen = torch.Generator().manual_seed(0)
    for _ in range(200):
        other = torch.softmax(torch.log(pi_b) + 0.3 * torch.randn(K, generator=gen), 0)
        assert rlhf_objective(other, true_reward, 0.5) <= best + 1e-6
    print(f"objective at pi*: {best:.4f}; 200 perturbed policies all score lower: ok")

    # KL-penalised policy improvement by gradient ascent reaches the same policy
    theta = log_pi_ref.clone().requires_grad_(True)
    opt = torch.optim.Adam([theta], lr=0.05)
    for _ in range(1500):
        loss = -rlhf_objective(torch.softmax(theta, 0), true_reward, 0.5)
        opt.zero_grad(); loss.backward(); opt.step()
    gap = (torch.softmax(theta, 0) - pi_b).abs().max().item()
    print(f"gradient ascent on reward - beta*KL lands on pi*: max |difference| = {gap:.1e}")
    assert gap < 1e-3

[TODO] kl_regularized_optimum is not implemented yet: skipping this check.


### Reward hacking, in one sweep over $\beta$

Now optimise the **learned** reward model $r_\phi$ with different $\beta$, and score each resulting policy with the **true** reward $r^*$, which the optimiser never sees. This mirrors Gao, Schulman & Hilton (2023), who measured how the true ("gold") reward behaves as a policy is optimised against a proxy reward model and moves away from the start in KL.

In [13]:
if pi_b is not None and "w_rm" in globals():
    betas = torch.logspace(-2, 1.5, 30)
    rows = []
    for beta in betas:
        pi = kl_regularized_optimum(log_pi_ref, rm_reward, beta)
        rows.append(((pi * rm_reward).sum().item(), (pi * true_reward).sum().item(),
                     kl_divergence(pi, log_pi_ref.exp()).item(), (pi * length).sum().item()))
    rm_r, true_r, kls, lens = map(np.array, zip(*rows))
    i_best = true_r.argmax()
    print(f"pi_ref:              RM reward {(log_pi_ref.exp() * rm_reward).sum():5.2f}, "
          f"true reward {(log_pi_ref.exp() * true_reward).sum():5.2f}, mean length {(log_pi_ref.exp() * length).sum():4.0f}")
    print(f"best beta = {betas[i_best]:.3f}: RM reward {rm_r[i_best]:5.2f}, true reward {true_r[i_best]:5.2f}, "
          f"KL {kls[i_best]:.2f}, mean length {lens[i_best]:4.0f}")
    print(f"beta = {betas[0]:.3f}:       RM reward {rm_r[0]:5.2f}, true reward {true_r[0]:5.2f}, "
          f"KL {kls[0]:.2f}, mean length {lens[0]:4.0f}")
    assert np.all(np.diff(rm_r) <= 1e-6)        # smaller beta -> more RM reward, always
    assert np.all(np.diff(kls) <= 1e-6)         # ... and more drift from pi_ref
    assert 0 < i_best < len(betas) - 1 and true_r[0] < true_r[i_best] - 0.3   # but the true reward peaks, then falls

    fig, ax = plt.subplots(figsize=(6, 3.6))
    ax.plot(kls, rm_r, "o-", ms=3, label=r"reward model $r_\phi$ (what we optimise)")
    ax.plot(kls, true_r, "o-", ms=3, label=r"true reward $r^*$ (what we want)")
    ax.set_xlabel(r"$D_{KL}(\pi^*_\beta \,\|\, \pi_{ref})$  (small $\beta$ = right)"); ax.legend()
    ax.set_title("over-optimising a learned reward")
    plt.tight_layout(); plt.show()

Read the plot from left to right ($\beta$ shrinking). The reward model's score rises the whole way; the asserts check that. The true reward first rises, then falls: the policy drifts to long answers the reward model over-rates. This is **reward hacking**. The KL term is the brake: a moderate $\beta$ keeps the policy where the reward model was trained and can be trusted.

## Part 6. DPO: the same objective as a classification loss

Rearranging the closed form gives $r(x, y) = \beta \log \frac{\pi(y\mid x)}{\pi_{\text{ref}}(y\mid x)} + \beta \log Z(x)$. Put this *implicit reward* into the Bradley–Terry loss; $\log Z(x)$ cancels in the difference. The result is the DPO loss (Rafailov et al., 2023):

$$\mathcal{L}_{\text{DPO}} = -\,\mathbb{E}\Big[\log \sigma\Big(\beta \log \tfrac{\pi_\theta(y_w\mid x)}{\pi_{\text{ref}}(y_w\mid x)} - \beta \log \tfrac{\pi_\theta(y_l\mid x)}{\pi_{\text{ref}}(y_l\mid x)}\Big)\Big].$$

No reward model, no sampling: just the preference pairs and two log-probabilities per answer.

In [14]:
# TODO 5: the DPO loss.
def dpo_loss(logp_w, logp_l, ref_logp_w, ref_logp_l, beta):
    """All inputs (N,): log pi_theta and log pi_ref of the preferred (w) and rejected (l) answers.
    Return the mean DPO loss. Hint: it is bt_loss applied to the implicit rewards beta * (log pi - log pi_ref)."""
    raise NotImplementedError

In [15]:
torch.manual_seed(2)
lw, ll, rw_, rl_ = torch.randn(4, 30)
d = attempt(dpo_loss, lw, ll, rw_, rl_, 0.1)
if d is not None:
    by_hand = -torch.log(torch.sigmoid(0.1 * ((lw - rw_) - (ll - rl_)))).mean()
    assert torch.isclose(d, by_hand, atol=1e-6)
    assert torch.isclose(dpo_loss(lw + 3.0, ll + 3.0, rw_, rl_, 0.1), d), "log Z(x) should cancel"
    assert torch.isclose(dpo_loss(rw_, rl_, rw_, rl_, 0.1), torch.tensor(math.log(2)))
    print("matches the formula; a per-prompt constant (log Z) cancels; pi = pi_ref gives log 2: ok")

    # train a table of logits over the K answers with DPO on the same preference pairs
    beta = 0.1
    theta = log_pi_ref.clone().requires_grad_(True)
    opt = torch.optim.Adam([theta], lr=0.02)
    for step in range(800):
        logp = torch.log_softmax(theta, 0)
        loss = dpo_loss(logp[YW], logp[YL], log_pi_ref[YW], log_pi_ref[YL], beta)
        opt.zero_grad(); loss.backward(); opt.step()
    pi_dpo = torch.softmax(theta, 0).detach()
    implicit = beta * (torch.log(pi_dpo) - log_pi_ref)
    acc = (implicit[YW_test] > implicit[YL_test]).float().mean().item()
    print(f"DPO policy: true reward {(pi_dpo * true_reward).sum():.2f} (pi_ref "
          f"{(log_pi_ref.exp() * true_reward).sum():.2f}); its implicit reward agrees with held-out "
          f"choices on {acc:.1%} of pairs; mean length {(pi_dpo * length).sum():.0f}")
    assert (pi_dpo * true_reward).sum() > (log_pi_ref.exp() * true_reward).sum()

[TODO] dpo_loss is not implemented yet: skipping this check.


DPO improved the true reward without ever fitting a separate reward model: the policy itself defines the reward. Note what it learns from: a fixed set of pairs, all drawn from $\pi_{\text{ref}}$. PPO and GRPO instead keep sampling new answers from the current policy. That online exploration is what the next part uses.

## Part 7. GRPO: the group is the baseline

Back to the arithmetic prompts and the verifier. For each prompt $x$, GRPO (Shao et al., 2024) samples a group of $G$ answers $y^{(1)}, \dots, y^{(G)}$ from $\pi_{\theta_{\text{old}}}$, scores them, and normalises the rewards **within the group**:

$$\hat A^{(g)} = \frac{r^{(g)} - \mathrm{mean}\big(r^{(1)}, \dots, r^{(G)}\big)}{\mathrm{std}\big(r^{(1)}, \dots, r^{(G)}\big) + \delta}.$$

The group mean plays the role of $V(s)$ from Part 3, so no critic is needed. The slides' example: $G = 4$, rewards $1, 0, 0, 1$ give mean 0.5, std 0.5, so $\hat A = +1, -1, -1, +1$. (Those numbers use the population std, dividing by $G$; we do the same.) If every answer in a group gets the same reward, $\delta$ makes all advantages 0: that group teaches nothing.

In [16]:
# TODO 6: group-relative advantages.
def group_advantages(rewards, delta=1e-6):
    """rewards: (P, G) tensor, one row per prompt, one column per sampled answer.
    Return the (P, G) advantages (r - row mean) / (row std + delta), with the population std
    (torch: unbiased=False)."""
    raise NotImplementedError

In [17]:
adv = attempt(group_advantages, torch.tensor([[1., 0., 0., 1.]]))
if adv is not None:
    print("rewards 1,0,0,1 ->", [round(v, 3) for v in adv.squeeze().tolist()])
    assert torch.allclose(adv, torch.tensor([[1., -1., -1., 1.]]), atol=1e-4)

    r = torch.tensor([[1., 1., 1., 0.]])                # the baseline check slide
    print("rewards 1,1,1,0 -> mean baseline only:", (r - r.mean()).squeeze().tolist(),
          " group-normalised:", [round(v, 3) for v in group_advantages(r).squeeze().tolist()])
    assert torch.allclose(r - r.mean(), torch.tensor([[0.25, 0.25, 0.25, -0.75]]))
    assert torch.allclose(group_advantages(r), (r - 0.75) / math.sqrt(0.1875), atol=1e-4)

    assert torch.all(group_advantages(torch.ones(3, 8)) == 0), "an all-equal group must give zero advantages"
    rr = torch.rand(5, 8)
    assert torch.allclose(group_advantages(rr), group_advantages(10 * rr + 3), atol=1e-4)
    print("all-equal groups give 0; invariant to shifting and scaling the rewards: ok")

[TODO] group_advantages is not implemented yet: skipping this check.


In the group 1, 1, 1, 0, the one wrong answer gets a large negative weight and each correct one a small positive weight: a rare failure carries more signal than a common success.

### The PPO clip

GRPO reuses each batch of samples for a few gradient steps, so the policy moves away from $\pi_{\theta_{\text{old}}}$ that produced them. PPO (Schulman et al., 2017) limits that move with the ratio $\rho = \pi_\theta / \pi_{\theta_{\text{old}}}$ of the action taken:

$$L^{\text{CLIP}} = \min\big(\rho\, \hat A,\ \mathrm{clip}(\rho, 1-\varepsilon, 1+\varepsilon)\, \hat A\big).$$

With $\hat A > 0$ there is no gain from raising $\rho$ above $1+\varepsilon$; with $\hat A < 0$ no gain from lowering it below $1 - \varepsilon$.

In [18]:
# TODO 7: the PPO clipped objective (to be maximised), elementwise.
def ppo_clip_objective(rho, adv, eps=0.2):
    """rho, adv: tensors of the same shape. Return min(rho * adv, clip(rho, 1-eps, 1+eps) * adv), elementwise.
    Hint: torch.clamp and torch.minimum."""
    raise NotImplementedError

In [19]:
rho = torch.tensor([1.5, 1.5, 0.5, 0.5, 1.0, 1.1], requires_grad=True)
A_ = torch.tensor([1.0, -1.0, 1.0, -1.0, 2.0, 1.0])
out = attempt(ppo_clip_objective, rho, A_)
if out is not None:
    print("rho:", [round(v, 3) for v in rho.tolist()], "\nA:  ", A_.tolist(), "\nL:  ", [round(v, 3) for v in out.tolist()])
    assert torch.allclose(out, torch.tensor([1.2, -1.5, 0.5, -0.8, 2.0, 1.1]))
    out.sum().backward()
    print("dL/drho:", rho.grad.tolist())
    # clipped entries (A>0 and rho too high; A<0 and rho too low) get no gradient
    assert torch.allclose(rho.grad, torch.tensor([0.0, -1.0, 1.0, 0.0, 2.0, 1.0]))

[TODO] ppo_clip_objective is not implemented yet: skipping this check.


### Training: three ways to weight $\nabla \log \pi$

All three runs start from $\pi_{\text{ref}}$ and see the same number of samples per step: 16 prompts × $G = 8$ answers. Only the weight $W$ changes, as in the slides' table:

1. **REINFORCE**, $W = r$.
2. **Actor-critic**, $W = r - V_w(x)$, where $V_w$ is a second small network (the critic) trained by regression on the rewards.
3. **GRPO**, $W = \hat A^{(g)}$ from your `group_advantages`, with the PPO clip ($\varepsilon = 0.2$, two gradient steps per batch) and a KL penalty $\beta\, D_{\mathrm{KL}}(\pi_\theta \| \pi_{\text{ref}})$. With one-token answers we can compute this KL exactly over the 19 tokens. (DeepSeekMath instead uses a per-token sampled estimate of it.)

Every token of an answer would share its answer's advantage; here an answer is one token.

In [20]:
P, G_SIZE, STEPS, LR, SEEDS = 16, 8, 300, 1e-2, [0, 1, 2]

def run(method, beta=0.04, steps=STEPS, seed=0):
    torch.manual_seed(seed)
    gen = torch.Generator().manual_seed(seed)
    pol = Policy(); pol.load_state_dict(ref_policy.state_dict())
    opt = torch.optim.Adam(pol.parameters(), lr=LR)
    critic = Policy() if method == "actor-critic" else None     # its output[:, 0] is used as V(x)
    copt = torch.optim.Adam(critic.parameters(), lr=LR) if critic else None
    hist, groups = [], []
    for step in range(steps + 1):
        if step % 20 == 0:
            hist.append((step, evaluate(pol, ref_policy)))
        if step == steps:
            break
        idx = torch.randint(100, (P,), generator=gen)
        x = PROMPTS[idx].repeat_interleave(G_SIZE, 0)          # each prompt G times
        with torch.no_grad():
            logp_old_all = F.log_softmax(pol(x), -1)
            y = torch.multinomial(logp_old_all.exp(), 1, generator=gen).squeeze(1)
            logp_old = logp_old_all[torch.arange(len(y)), y]
        r = verify(x, y)
        rg = r.view(P, G_SIZE)
        groups.append(((rg.sum(1) == G_SIZE).float().mean().item(), (rg.sum(1) == 0).float().mean().item()))
        if method == "reinforce":
            loss = reinforce_surrogate(pol(x), y, r)
            opt.zero_grad(); loss.backward(); opt.step()
        elif method == "actor-critic":
            V = critic(x)[:, 0]
            closs = F.mse_loss(V, r)
            copt.zero_grad(); closs.backward(); copt.step()
            loss = reinforce_surrogate(pol(x), y, r - V.detach())
            opt.zero_grad(); loss.backward(); opt.step()
        else:  # grpo
            adv = group_advantages(rg).flatten()
            ref_probs = F.softmax(ref_policy(x), -1)
            for _ in range(2):                                   # reuse the batch: PPO-style steps
                logp_all = F.log_softmax(pol(x), -1)
                rho = torch.exp(logp_all[torch.arange(len(y)), y] - logp_old)
                kl = kl_divergence(logp_all.exp(), ref_probs)
                loss = -(ppo_clip_objective(rho, adv, 0.2) - beta * kl).mean()
                opt.zero_grad(); loss.backward(); opt.step()
    return hist, np.array(groups)

def run_seeds(method, **kw):
    # average the evaluation curves over SEEDS
    runs = [run(method, seed=s, **kw) for s in SEEDS]
    steps = [st for st, _ in runs[0][0]]
    curve = {k: np.mean([[e[k] for _, e in h] for h, _ in runs], 0) for k in runs[0][0][0][1]}
    return steps, curve, np.mean([g for _, g in runs], 0)

READY = all(attempt(f, *a) is not None for f, a in [
    (reinforce_surrogate, (torch.zeros(1, VOCAB), torch.zeros(1, dtype=torch.long), torch.ones(1))),
    (group_advantages, (torch.ones(1, 2),)),
    (ppo_clip_objective, (torch.ones(1), torch.ones(1)))])
if READY:
    t0 = time.time()
    results = {m: run_seeds(m) for m in ["reinforce", "actor-critic", "grpo"]}
    print(f"{3 * len(SEEDS)} runs in {time.time() - t0:.0f} s; numbers are means over seeds {SEEDS}\n")
    print(f"{'method':14s} {'reward @100':>12s} {'reward @end':>12s} {'greedy acc':>11s} {'KL to ref':>10s}  (end = step {STEPS})")
    print(f"{'pi_ref':14s} {REF_EVAL['expected reward']:12.3f} {REF_EVAL['expected reward']:12.3f} "
          f"{REF_EVAL['greedy acc']:11.3f} {0:10.3f}")
    i100 = results["grpo"][0].index(100)
    for m, (st, c, _) in results.items():
        print(f"{m:14s} {c['expected reward'][i100]:12.3f} {c['expected reward'][-1]:12.3f} "
              f"{c['greedy acc'][-1]:11.3f} {c['KL to ref'][-1]:10.3f}")
    gr = results["grpo"][2]
    print(f"\nGRPO groups with zero advantage (all {G_SIZE} answers equal):")
    print(f"  first 10 steps: all correct {gr[:10, 0].mean():.2f}, all wrong {gr[:10, 1].mean():.2f}")
    print(f"  last 10 steps:  all correct {gr[-10:, 0].mean():.2f}, all wrong {gr[-10:, 1].mean():.2f}")

    fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
    for m, (st, c, _) in results.items():
        ax[0].plot(st, c["expected reward"], "o-", ms=3, label=m)
        ax[1].plot(st, c["KL to ref"], "o-", ms=3, label=m)
    ax[0].set_title("expected reward (verifier)"); ax[1].set_title(r"KL($\pi_\theta \| \pi_{ref}$)")
    for a_ in ax: a_.set_xlabel("step"); a_.legend()
    plt.tight_layout(); plt.show()
    for m, (st, c, _) in results.items():
        assert c["expected reward"][-1] > REF_EVAL["expected reward"] + 0.4, m
    assert gr[-10:, 0].mean() > gr[:10, 0].mean() and gr[:10, 1].mean() > gr[:10, 0].mean()

[TODO] reinforce_surrogate is not implemented yet: skipping this check.


All three methods learn from the verifier alone: after the noisy demonstrations, nobody showed the policy a correct answer. Compare the rows: plain REINFORCE against the two baselines (a learned critic, or the group mean), early (step 100) and at the end. Remember the cost that the table does not show: actor-critic trains a second network as large as the policy, while GRPO gets its baseline from the group for free.

Look at the zero-advantage groups too. Early on, the groups that teach nothing are mostly all-wrong; late in training, more and more groups are all-correct. Either way those prompts give no gradient. Real GRPO pipelines face the same problem with prompts that are too hard or too easy.

### KL drift: what $\beta$ does

With a verifier there is no reward model to hack, but the policy still drifts from $\pi_{\text{ref}}$. We run GRPO with three values of $\beta$ and compare reward and KL at the end.

In [21]:
if READY:
    drift = {}
    for beta in [0.0, 0.1, 1.0]:
        st, c, _ = run_seeds("grpo", beta=beta)
        drift[beta] = (c["expected reward"][-1], c["KL to ref"][-1])
        print(f"beta = {beta:3.1f}: expected reward {drift[beta][0]:.3f}, KL to ref {drift[beta][1]:.3f}")
    assert drift[0.0][1] > drift[0.1][1] > drift[1.0][1]
    assert drift[1.0][0] < min(drift[0.0][0], drift[0.1][0])
    print("a larger beta keeps the policy closer to pi_ref; the largest beta also costs reward: ok")

A larger $\beta$ holds the policy nearer to $\pi_{\text{ref}}$, and the largest one gives up reward to do it. In Part 5 that trade was what saved us from a flawed reward model. With an exact verifier the KL term protects against less, so its weight is a design choice.

One caution if you compare with Part 5: GRPO divides rewards by the group's std, so the reward scale inside its objective changes from group to group. Its $\beta$ is therefore not on the same scale as the $\beta$ in $\pi_{\text{ref}}\, e^{r/\beta}$.

## Summary

| Part | Idea | What you computed |
|---|---|---|
| 1 | Return $G_t = r_t + \gamma G_{t+1}$ | the grid-world values 100, 50, 25, 12.5 and the 6.25 detour |
| 2 | An LLM is a policy; a verifier is a reward | a one-token "arithmetic LLM" and an SFT-like reference |
| 3 | Policy gradient = weighted $\nabla \log \pi$ of own samples | the surrogate loss; $W = 1$ gives cross-entropy; a baseline keeps the mean and cuts the variance |
| 4 | Bradley–Terry reward model | fit on pairs; it learned "longer is better" from short answers only |
| 5 | Reward minus $\beta \cdot$ KL | the closed-form optimum; reward hacking as $\beta \to 0$ |
| 6 | DPO | the same objective as a classification loss on pairs |
| 7 | GRPO | group-normalised advantages, the PPO clip, and $\beta$ controlling KL drift |

**Credits.** Policy gradient and REINFORCE: R. J. Williams, *Machine Learning* 8 (1992). Returns, baselines and actor-critic: R. Sutton & A. Barto, *Reinforcement Learning: An Introduction*, 2nd ed. (MIT Press, 2018); the 3×3 grid-world example after P. Bodik (UC Berkeley) as used on the slides. Pairwise preference model: R. A. Bradley & M. E. Terry, *Biometrika* (1952); reward models from human preferences: P. Christiano et al., NeurIPS 2017; RLHF with a KL penalty to a reference policy: D. Ziegler et al., arXiv 2019, and L. Ouyang et al. (InstructGPT), NeurIPS 2022. Reward-model over-optimisation: L. Gao, J. Schulman & J. Hilton, ICML 2023. PPO: J. Schulman et al., arXiv 2017. DPO and the closed-form KL-regularised optimum: R. Rafailov et al., NeurIPS 2023. GRPO: Z. Shao et al. (DeepSeekMath), arXiv 2024; DeepSeek-AI, DeepSeek-R1, 2025. The arithmetic task, the synthetic preference data and all numbers in this notebook are made up for teaching.

In [22]:
print(f"Total notebook runtime: {time.time() - T_START:.0f} s")

Total notebook runtime: 0 s
